# Brain MRI analysis in Colab (CPTAC-GBM / BraTS / any NIfTI)

Run the cells top to bottom with **Shift+Enter**. What you get:

| Part | What it shows | Needs |
| --- | --- | --- |
| A. Setup | installs tools, optional Google Drive | nothing |
| B. Dataset overview | charts of all MRI series in CPTAC-GBM: sequences, complete patients, scanners, slice counts | internet only (metadata) |
| C. Download | one patient, converted to `.nii.gz`, size-limited | your TCIA access approval |
| D. Scan analysis | header table, 4 sequences side by side, interactive slice viewer, montage, histograms, brain mask, quality checks, 3D view | downloaded files **or demo mode** |
| E. Report | summary table + figures saved to your folder | part D |

**No data yet?** Leave `DEMO = True` in part D. It uses a real 3D brain MRI template (MNI152) so you can try every visual now.

## A. Setup

In [ ]:
!pip install -q tcia_utils dcm2niix nibabel nilearn plotly ipywidgets scikit-image
import os, glob, warnings
warnings.filterwarnings('ignore')
SAVE_TO_DRIVE = False        # True = keep files in Google Drive between sessions
WORK = '/content/phd_data'
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    WORK = os.path.abspath('phd_data')
if SAVE_TO_DRIVE and IN_COLAB:
    from google.colab import drive, output
    drive.mount('/content/drive'); WORK = '/content/drive/MyDrive/phd_data'
if IN_COLAB:
    from google.colab import output; output.enable_custom_widget_manager()
os.makedirs(WORK, exist_ok=True); os.chdir(WORK)
FIG = os.path.join(WORK, 'figures'); os.makedirs(FIG, exist_ok=True)
print('working in', WORK)

In [ ]:
%%writefile tcia_cptac_gbm.py
"""List and download brain MRI series from TCIA's CPTAC-GBM collection with the TCIA API.

Run on your own computer (needs internet):
    pip install tcia_utils pandas
    python tcia_cptac_gbm.py --list                 # step 1: see which MRI series exist per patient
    python tcia_cptac_gbm.py --download --patients 1 --max-mb 300 --convert
        # step 2: one patient's T1/T1c/T2/FLAIR, refuse if over 300 MB, convert to .nii.gz, delete the DICOMs
    (--convert needs:  pip install dcm2niix)

Note: CPTAC-GBM radiology is under NIH controlled access. Listing metadata may work without approval,
but downloading images is expected to fail (or return nothing) until your access request is approved.
If the API refuses downloads, use step 1's CSV to pick series and download them with the TCIA Data Retriever.
"""
import argparse
import os
import re
import shutil
import subprocess

import pandas as pd
from tcia_utils import nbia

COLLECTION = "CPTAC-GBM"
# Rough patterns for the four BraTS sequences in free-text SeriesDescription fields (check them by eye!)
PATTERNS = {
    "FLAIR": r"flair",
    "T1c": r"(t1.*(post|\+c|gad|gd|contrast))|((post|gad|gd).*(t1|mprage))|(mprage.*(post|gad|gd|\+c))",
    "T2": r"\bt2\b(?!.*flair)|t2[ _-]?(tse|fse|ax|axial|space)",
    "T1": r"\bt1\b|t1[ _-]?(se|ax|axial|mprage|flash|tfe)|mprage|spgr",
}
SKIP = r"loc|scout|survey|calib|dwi|diff|adc|trace|perf|dsc|dce|swi|mra|tof|spect|cbv|fa\b"


def guess(desc: str) -> str:
    d = (desc or "").lower()
    if re.search(SKIP, d):
        return "other"
    for name, pat in PATTERNS.items():  # order matters: T1c before T1
        if re.search(pat, d):
            return name
    return "other"


def list_series() -> pd.DataFrame:
    df = nbia.getSeries(collection=COLLECTION, modality="MR", format="df")
    if df is None or len(df) == 0:
        raise SystemExit("No series returned. The API may require approved access for this collection.")
    df["sequence_guess"] = df["SeriesDescription"].fillna("").map(guess)
    cols = [c for c in ["PatientID", "StudyInstanceUID", "SeriesInstanceUID", "SeriesDescription",
                        "sequence_guess", "ImageCount", "FileSize", "Manufacturer"] if c in df.columns]
    df = df[cols].sort_values(["PatientID", "sequence_guess"])
    df.to_csv("cptac_gbm_mr_series.csv", index=False)  # same file name for any collection
    have = df[df.sequence_guess != "other"].groupby("PatientID")["sequence_guess"].agg(lambda s: sorted(set(s)))
    complete = have[have.map(len) == 4]
    print(f"{df.PatientID.nunique()} patients, {len(df)} MR series  ->  saved cptac_gbm_mr_series.csv")
    print(f"{len(complete)} patients appear to have all four of T1, T1c, T2, FLAIR")
    print(df.SeriesDescription.value_counts().head(25).to_string())
    return df


def convert_and_clean(out: str):
    """Convert every DICOM series folder under `out` to compressed NIfTI, then delete the DICOM files."""
    exe = shutil.which("dcm2niix")
    if not exe:
        print("dcm2niix not found (pip install dcm2niix); keeping DICOMs.")
        return
    nii = os.path.join(out, "nifti")
    os.makedirs(nii, exist_ok=True)
    series_dirs = {r for r, _, fs in os.walk(out) if "nifti" not in r and any(f.lower().endswith(".dcm") for f in fs)}
    for d in sorted(series_dirs):
        r = subprocess.run([exe, "-z", "y", "-f", "%i_%d_%s", "-o", nii, d], capture_output=True, text=True)
        if r.returncode == 0:
            shutil.rmtree(d)
        else:
            print("conversion failed, kept:", d)
    mb = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(nii) for f in fs) / 1e6
    print(f"NIfTI files in {nii}/  ({mb:.0f} MB total)")


def download(n_patients: int, out: str, max_mb: float, convert: bool):
    df = list_series()
    df = df[df.sequence_guess != "other"]
    full = df.groupby("PatientID")["sequence_guess"].nunique()
    pats = full[full == 4].index[:n_patients]
    # one series per sequence per patient: the one with the most slices
    if "ImageCount" in df.columns:
        df = df.sort_values("ImageCount", ascending=False)
    pick = df[df.PatientID.isin(pats)].drop_duplicates(["PatientID", "sequence_guess"])
    print(pick[["PatientID", "sequence_guess", "SeriesDescription"] + (["ImageCount"] if "ImageCount" in pick else [])].to_string(index=False))
    if "FileSize" in pick.columns:
        mb = pd.to_numeric(pick.FileSize, errors="coerce").sum() / 1e6
        print(f"Estimated download: {mb:.0f} MB for {len(pick)} series (patients: {list(pats)})")
        if mb > max_mb:
            raise SystemExit(f"Over your --max-mb limit of {max_mb:.0f} MB. Use fewer patients or raise the limit.")
    nbia.downloadSeries(pick, input_type="df", path=out, csv_filename="downloaded_series")
    print(f"Downloaded -> {out}/")
    if convert:
        convert_and_clean(out)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--collection", default="CPTAC-GBM", help="e.g. UPENN-GBM (open access)")
    ap.add_argument("--list", action="store_true")
    ap.add_argument("--download", action="store_true")
    ap.add_argument("--patients", type=int, default=1)
    ap.add_argument("--max-mb", type=float, default=300, help="refuse to download more than this")
    ap.add_argument("--convert", action="store_true", help="convert to .nii.gz and delete the DICOMs")
    ap.add_argument("--out", default="cptac_gbm_dicom")
    a = ap.parse_args()
    COLLECTION = a.collection
    if a.download:
        download(a.patients, a.out, a.max_mb, a.convert)
    else:
        list_series()


## B. Dataset overview (metadata only, a few MB)

Lists every MRI series in the chosen collection and charts what is there, before you download any images.

In [ ]:
COLLECTION = 'UPENN-GBM'   # open access now; switch to 'CPTAC-GBM' once your access is approved
RUN_LIST = True
if RUN_LIST:
    !python -W ignore tcia_cptac_gbm.py --list --collection {COLLECTION}

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})
CSV = 'cptac_gbm_mr_series.csv'
if not os.path.exists(CSV):
    print('No series list yet (step above failed or was skipped). Part B is skipped.')
else:
    df = pd.read_csv(CSV)
    seqs = ['T1', 'T1c', 'T2', 'FLAIR']
    per_pat = df[df.sequence_guess.isin(seqs)].groupby('PatientID').sequence_guess.agg(lambda s: set(s))
    n_have = per_pat.map(len).reindex(df.PatientID.unique(), fill_value=0)
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
    df.sequence_guess.value_counts().reindex(seqs + ['other']).plot.bar(ax=ax[0], color=['C0','C1','C2','C3','0.6'])
    ax[0].set_title('MR series by sequence (guessed)'); ax[0].set_ylabel('series')
    n_have.value_counts().sort_index().plot.bar(ax=ax[1], color='C4')
    ax[1].set_title(f'How many of T1/T1c/T2/FLAIR each patient has\n({(n_have==4).sum()} of {len(n_have)} have all four)')
    ax[1].set_xlabel('number of the 4 sequences'); ax[1].set_ylabel('patients')
    if 'ImageCount' in df:
        for s in seqs:
            v = df.loc[df.sequence_guess == s, 'ImageCount']
            if len(v): ax[2].hist(v, bins=30, alpha=0.6, label=s)
        ax[2].legend(); ax[2].set_title('Slices per series (few slices = thick slices)'); ax[2].set_xlabel('images in series')
    plt.tight_layout(); plt.savefig(f'{FIG}/dataset_overview.png'); plt.show()
    if 'Manufacturer' in df:
        df.drop_duplicates('PatientID').Manufacturer.fillna('unknown').value_counts().plot.barh(figsize=(7, 3), color='C5')
        plt.title('Scanner manufacturer (per patient)'); plt.tight_layout(); plt.show()
    # patient x sequence availability grid (first 40 patients)
    grid = pd.DataFrame({s: per_pat.map(lambda x: s in x) for s in seqs}).reindex(n_have.index, fill_value=False).astype(int)
    g = grid.head(40)
    plt.figure(figsize=(4, max(3, len(g) * 0.18)))
    plt.imshow(g.values, cmap='Greens', aspect='auto', vmin=0, vmax=1)
    plt.xticks(range(4), seqs); plt.yticks(range(len(g)), g.index, fontsize=6)
    plt.title('Sequence availability (green = present)'); plt.tight_layout(); plt.show()
    display(df.SeriesDescription.value_counts().head(20).rename('series').to_frame())

## C. Download one patient

Stops if the download is above `MAX_MB`. Converts to `.nii.gz` and deletes the DICOMs to save space.

In [ ]:
RUN_DOWNLOAD = True    # UPENN-GBM is open; CPTAC-GBM needs approval
MAX_MB = 300
if RUN_DOWNLOAD:
    !python -W ignore tcia_cptac_gbm.py --download --collection {COLLECTION} --patients 1 --max-mb {MAX_MB} --convert

## D. Scan analysis

Set `DEMO = False` and `DATA_DIR` to your folder of `.nii.gz` files (from part C, or a BraTS case folder).

In [ ]:
DEMO = True
DATA_DIR = 'cptac_gbm_dicom/nifti'      # or e.g. 'BraTS-GLI-00000-000'

import nibabel as nib
from scipy import ndimage as ndi
from skimage.filters import threshold_otsu

def label_for(path):
    n = os.path.basename(path).lower()
    for key, lab in [('flair', 'FLAIR'), ('t2f', 'FLAIR'), ('t1c', 'T1c'), ('t1ce', 'T1c'), ('post', 'T1c'), ('gad', 'T1c'),
                     ('t2', 'T2'), ('t1', 'T1'), ('seg', 'SEG')]:
        if key in n: return lab
    return os.path.basename(path).split('.')[0][:20]

if DEMO:
    from nilearn import datasets as _nd
    d = os.path.join(os.path.dirname(_nd.__file__), 'data')
    files = [f'{d}/mni_icbm152_t1_tal_nlin_sym_09a_converted.nii.gz']
    print('DEMO mode: MNI152 T1 template (a real 3D MRI, one sequence, already skull-stripped)')
else:
    files = sorted(glob.glob(os.path.join(DATA_DIR, '*.nii*')))
    print(f'{len(files)} files in {DATA_DIR}')

scans = {}
for f in files:
    im = nib.as_closest_canonical(nib.load(f))        # reorient to RAS so all scans display the same way
    scans[label_for(f)] = {'path': f, 'img': im, 'vol': im.get_fdata(dtype=np.float32)}
print('loaded:', list(scans))

### D1. Header table: size, voxel spacing, orientation, intensities

In [ ]:
rows = []
for k, s in scans.items():
    v, z = s['vol'], s['img'].header.get_zooms()[:3]
    rows.append({'scan': k, 'shape': v.shape[:3], 'voxel mm': tuple(np.round(z, 2)),
                 'field of view mm': tuple(np.round(np.array(v.shape[:3]) * z)),
                 'slice thickness mm': round(float(max(z)), 2), 'min': round(float(v.min()), 1),
                 'max': round(float(v.max()), 1), 'file MB': round(os.path.getsize(s['path']) / 1e6, 1)})
hdr = pd.DataFrame(rows); display(hdr)
for r in rows:
    if r['slice thickness mm'] > 2.5:
        print(f"! {r['scan']}: {r['slice thickness mm']} mm slices - thick; 3D detail between slices is limited")

### D2. All sequences side by side (middle slice, three planes)

In [ ]:
def mid(v): return [s // 2 for s in v.shape[:3]]
def norm(sl):
    lo, hi = np.percentile(sl[sl > 0], [1, 99]) if (sl > 0).any() else (0, 1)
    return np.clip((sl - lo) / (hi - lo + 1e-6), 0, 1)
img_scans = {k: s for k, s in scans.items() if k != 'SEG'}
fig, ax = plt.subplots(len(img_scans), 3, figsize=(11, 3.4 * len(img_scans)), squeeze=False)
for r, (k, s) in enumerate(img_scans.items()):
    v = s['vol']; x, y, z = mid(v)
    for c, (sl, name) in enumerate([(v[:, :, z], 'axial'), (v[:, y, :], 'coronal'), (v[x, :, :], 'sagittal')]):
        ax[r, c].imshow(np.rot90(norm(sl)), cmap='gray'); ax[r, c].axis('off')
        ax[r, c].set_title(f'{k} - {name}', fontsize=10)
plt.tight_layout(); plt.savefig(f'{FIG}/sequences_3planes.png'); plt.show()

### D3. Interactive slice viewer
Drag the slider and switch plane or sequence. (In Colab this updates live.)

In [ ]:
import ipywidgets as W
def view(scan, plane, pos):
    v = img_scans[scan]['vol']
    n = v.shape[{'axial': 2, 'coronal': 1, 'sagittal': 0}[plane]]
    i = int(pos / 100 * (n - 1))
    sl = {'axial': v[:, :, i], 'coronal': v[:, i, :], 'sagittal': v[i, :, :]}[plane]
    plt.figure(figsize=(5.5, 5.5)); plt.imshow(np.rot90(norm(sl)), cmap='gray')
    if 'SEG' in scans and scans['SEG']['vol'].shape == v.shape:
        sg = scans['SEG']['vol']; m = {'axial': sg[:, :, i], 'coronal': sg[:, i, :], 'sagittal': sg[i, :, :]}[plane]
        plt.imshow(np.rot90(np.ma.masked_equal(m, 0)), cmap='autumn', alpha=0.5)
    plt.title(f'{scan} {plane} slice {i}/{n-1}'); plt.axis('off'); plt.show()
W.interact(view, scan=list(img_scans), plane=['axial', 'coronal', 'sagittal'],
           pos=W.IntSlider(50, 0, 100, 1, description='position %'));

### D4. Montage: scroll through the whole volume at a glance

In [ ]:
k0 = list(img_scans)[0]; v = img_scans[k0]['vol']
zs = np.linspace(int(v.shape[2] * 0.15), int(v.shape[2] * 0.85), 16).astype(int)
fig, ax = plt.subplots(2, 8, figsize=(16, 4.5))
for a, z in zip(ax.ravel(), zs):
    a.imshow(np.rot90(norm(v[:, :, z])), cmap='gray'); a.set_title(f'z={z}', fontsize=8); a.axis('off')
fig.suptitle(f'{k0}: axial montage'); plt.tight_layout(); plt.savefig(f'{FIG}/montage.png'); plt.show()

### D5. Brain mask, intensity histograms and normalization

If the background is already zero (skull-stripped data such as BraTS), the mask is simply the non-zero voxels.
Clinical scans still include the skull; then a rough head mask from Otsu thresholding plus cleanup is enough for intensity statistics.
(For real skull-stripping use HD-BET or SynthStrip.)

In [ ]:
def rough_mask(v):
    if (v == 0).mean() > 0.3:      # background already zero (skull-stripped, e.g. BraTS): mask = non-zero voxels
        return ndi.binary_fill_holes(v > 0)
    t = threshold_otsu(v[v > 0])
    m = ndi.binary_opening(v > t, iterations=2)
    lab, n = ndi.label(m)
    if n: m = lab == (np.bincount(lab.ravel())[1:].argmax() + 1)     # keep largest component
    return ndi.binary_fill_holes(ndi.binary_closing(m, iterations=3))
for k, s in img_scans.items():
    s['mask'] = rough_mask(s['vol'])
    v, m = s['vol'], s['mask']
    zs_ = np.where(m, (v - v[m].mean()) / (v[m].std() + 1e-6), 0)
    fig, ax = plt.subplots(1, 4, figsize=(17, 3.6))
    x, y, z = mid(v)
    ax[0].imshow(np.rot90(norm(v[:, :, z])), cmap='gray'); ax[0].contour(np.rot90(m[:, :, z]), [0.5], colors='lime')
    ax[0].set_title(f'{k}: mask outline'); ax[0].axis('off')
    ax[1].hist(v[v > 0].ravel(), 120, color='0.5'); ax[1].set_title('all non-zero voxels'); ax[1].set_yscale('log')
    ax[2].hist(v[m].ravel(), 120, color='C1'); ax[2].set_title('inside mask (tissue peaks)')
    ax[3].hist(zs_[m].ravel(), 120, color='C2'); ax[3].set_title('after z-score normalization')
    plt.tight_layout(); plt.savefig(f'{FIG}/intensity_{k}.png'); plt.show()

### D6. Quality checks

- **SNR estimate**: mean signal inside the mask / standard deviation of the background air. Higher is better; compare scans relative to each other.
- **Slice profile**: mean intensity per axial slice. A sudden spike or drop points to a corrupt or missing slice, or motion.
- **Bias field hint**: intensity drift across the mask in one direction suggests the scanner's uneven coil sensitivity (corrected with N4).

In [ ]:
qc = []
fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
for k, s in img_scans.items():
    v, m = s['vol'], s['mask']
    bg = v[~ndi.binary_dilation(m, iterations=8) & (v > 0)]
    if bg.size < 100: bg = v[~ndi.binary_dilation(m, iterations=8)]
    snr = v[m].mean() / bg.std() if bg.size and bg.std() > 0 else np.nan   # nan: background is exactly zero
    area = m.sum((0, 1)); ok = area > 0.2 * area.max()          # ignore tiny edge slices
    prof = np.array([v[:, :, z][m[:, :, z]].mean() if ok[z] else np.nan for z in range(v.shape[2])])
    d = np.abs(np.diff(prof)) / np.nanmedian(prof)
    jumps = int(np.nansum(d > 0.10))                              # >10% change between neighbouring slices
    xs = np.nonzero(m)[0]; lo, hi = np.percentile(xs, [10, 90])
    drift = (v[(np.indices(v.shape)[0] > hi) & m].mean() - v[(np.indices(v.shape)[0] < lo) & m].mean()) / v[m].mean()
    qc.append({'scan': k, 'SNR (approx)': round(float(snr), 1), 'slice jumps': jumps, 'left-right drift %': round(100 * float(drift), 1),
               'mask volume ml': round(float(m.sum() * np.prod(s['img'].header.get_zooms()[:3]) / 1000))})
    ax[0].plot(prof / np.nanmax(prof), label=k)
    ax[1].bar(k, 0 if np.isnan(snr) else snr)
ax[0].set_title('mean intensity per axial slice (normalized)'); ax[0].set_xlabel('slice'); ax[0].legend()
ax[1].set_title('approximate SNR (0 = not measurable: background is zero)')
plt.tight_layout(); plt.savefig(f'{FIG}/quality.png'); plt.show()
qc = pd.DataFrame(qc); display(qc)

### D7. 3D view
An interactive surface of the head mask (rotate and zoom with the mouse). Downsampled for speed.

In [ ]:
import plotly.graph_objects as go
from skimage.measure import marching_cubes
k0 = list(img_scans)[0]; s = img_scans[k0]
step = max(1, int(np.ceil(max(s['mask'].shape) / 96)))
m = ndi.gaussian_filter(s['mask'][::step, ::step, ::step].astype(float), 1)
verts, faces, _, _ = marching_cubes(m, 0.5, spacing=tuple(np.array(s['img'].header.get_zooms()[:3]) * step))
traces = [go.Mesh3d(x=verts[:, 0], y=verts[:, 1], z=verts[:, 2], i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
                    color='lightgray', opacity=0.25, name='head/brain')]
if 'SEG' in scans:
    sg = (scans['SEG']['vol'][::step, ::step, ::step] > 0).astype(float)
    if sg.any():
        v2, f2, _, _ = marching_cubes(ndi.gaussian_filter(sg, 1), 0.5, spacing=tuple(np.array(s['img'].header.get_zooms()[:3]) * step))
        traces.append(go.Mesh3d(x=v2[:, 0], y=v2[:, 1], z=v2[:, 2], i=f2[:, 0], j=f2[:, 1], k=f2[:, 2], color='red', opacity=0.8, name='tumour'))
fig3d = go.Figure(traces); fig3d.update_layout(scene_aspectmode='data', height=550, title=f'{k0}: 3D surface (mm)')
fig3d.write_html(f'{FIG}/surface_3d.html'); fig3d.show()

## E. Report

In [ ]:
report = hdr.merge(qc, on='scan')
report.to_csv(f'{FIG}/scan_report.csv', index=False)
display(report)
print('Figures and report saved in', FIG); print('\n'.join(sorted(os.listdir(FIG))))
if IN_COLAB and not SAVE_TO_DRIVE:
    print('Tip: download the figures folder from the Files panel on the left, or set SAVE_TO_DRIVE = True.')